# 09 Eval Downstream

Evaluate learned and baseline embeddings/features on retention targets.

In [1]:
from pathlib import Path
import dataclasses
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.eval_downstream import evaluate_downstream, load_downstream_eval_config
base_config = load_downstream_eval_config(PROJECT_ROOT / "configs" / "eval_downstream.yaml")
base_config

DownstreamEvalConfig(inputs={'main_cls': 'data/exports/main_embeddings_cls.parquet', 'main_mean': 'data/exports/main_embeddings_mean.parquet', 'baseline_features': 'data/exports/baseline_features.parquet', 'baseline_embeddings': 'data/exports/baseline_embeddings_128d.parquet'}, outputs={'metrics': 'artifacts/reports/downstream_metrics.csv', 'manifest': 'artifacts/manifests/downstream_eval_manifest.json'}, targets=('retention_7d', 'retention_14d'), random_state=42, dry_run_rows=None)

In [3]:
DRY_RUN = True
config = base_config
if DRY_RUN:
    config = dataclasses.replace(config, dry_run_rows=2000)
print(config)

DownstreamEvalConfig(inputs={'main_cls': 'data/exports/main_embeddings_cls.parquet', 'main_mean': 'data/exports/main_embeddings_mean.parquet', 'baseline_features': 'data/exports/baseline_features.parquet', 'baseline_embeddings': 'data/exports/baseline_embeddings_128d.parquet'}, outputs={'metrics': 'artifacts/reports/downstream_metrics.csv', 'manifest': 'artifacts/manifests/downstream_eval_manifest.json'}, targets=('retention_7d', 'retention_14d'), random_state=42, dry_run_rows=2000)


In [4]:
manifest = evaluate_downstream(config, PROJECT_ROOT)
manifest

{'created_at': '2026-05-10T14:14:24.123615+00:00',
 'rows': 72,
 'targets': ['retention_7d', 'retention_14d'],
 'dry_run_rows': 2000,
 'feature_columns': ['prefix_len',
  'num_events_total',
  'next_event_token_id',
  'unique_event_count',
  'event_entropy',
  'repeat_count',
  'repeat_rate',
  'session_boundary_count',
  'gap_mean',
  'gap_std',
  'gap_max',
  'last_event_1',
  'last_event_2',
  'last_event_3',
  'last_event_4',
  'last_event_5',
  'markov_last_total_count',
  'markov_top1_prob',
  'markov_top10_entropy',
  'markov_actual_prob',
  'markov_actual_rank'],
 'outputs': {'metrics': 'artifacts/reports/downstream_metrics.csv',
  'manifest': 'artifacts/manifests/downstream_eval_manifest.json'}}

In [5]:
import pandas as pd
pd.read_csv(PROJECT_ROOT / config.outputs["metrics"])

,model_name,target,prefix_len,split,num_rows,positive_rate,roc_auc,pr_auc,f1,logloss
0,baseline_features_hgb,retention_14d,50,test,203,0.039409,0.846154,0.252759,0.315789,0.218370
1,baseline_features_hgb,retention_14d,50,valid,210,0.042857,0.945826,0.380085,0.300000,0.162492
2,baseline_features_logreg,retention_14d,50,test,203,0.039409,0.785256,0.466394,0.250000,0.373648
3,baseline_features_logreg,retention_14d,50,valid,210,0.042857,0.896075,0.351331,0.300000,0.355590
4,baseline_svd_logreg,retention_14d,50,test,203,0.039409,0.817949,0.286603,0.222222,0.235250
...,...,...,...,...,...,...,...,...,...,...
67,combined_cls_baseline_svd_logreg,retention_7d,150,valid,124,0.137097,0.617922,0.326027,0.275862,0.729653
68,main_cls_logreg,retention_7d,150,test,116,0.112069,0.546677,0.153384,0.214286,0.717187
69,main_cls_logreg,retention_7d,150,valid,124,0.137097,0.402969,0.179969,0.166667,1.256602
70,main_mean_logreg,retention_7d,150,test,116,0.112069,0.457804,0.120084,0.066667,1.137795
